In [ ]:
import numpy as np
from astropy import units as u
import csv
import configparser
import os
import subprocess

import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300
from matplotlib.pyplot import figure, show, subplots
from matplotlib import pyplot as plt

#import yaml
from ruamel.yaml import YAML
yaml = YAML()
yaml.preserve_quotes = True  # Keep quotes if they exist
yaml.indent(mapping=2, sequence=4, offset=2)  # Preserve indentation
yaml.representer.add_representer(
    float, lambda dumper, data: dumper.represent_scalar("tag:yaml.org,2002:float", str(data))
)

with open("../config/masterconfig.yaml", "r") as file:
    config = yaml.load(file) or {}
integrationtimes = config['integrationtimes']
dataserver_path = config['dataserver_path']

In [ ]:
def arcis_in_lifesim(arcisfolder,integrationtime):
    """Take one ARCiS spectrum, and run it through LIFEsim"""

    distance = arcisfolder.replace("fm_ARCiS", "").split("_")[-1]

    # convert the ARCiS emis file to go into LIFEsim
    folder = dataserver_path + '/ARCiS_forwardmodels/' + arcisfolder + "/emis" 
    bandcenter = np.genfromtxt(folder,skip_header=1,usecols=0)
    bandflux = np.genfromtxt(folder,skip_header=1,usecols=1)

    file_destination = "../output/LIFEsim/clean_spectra/textfiles/" + arcisfolder
    with open(file_destination, 'w') as f:
        writer = csv.writer(f, delimiter='\t')
        writer.writerows(zip(bandcenter,bandflux))

    # Edit LIFEsim config
    config = configparser.ConfigParser(comment_prefixes="#",allow_no_value=True)
    config.read('GEMINI/init_simulation.cfg')
    config['simulation']['simulation_distance-to-planet'] = str(distance)
    config['simulation']['simulation_distance-to-star'] = str(distance)
    config['simulation']['simulation_integration-time'] = str(integrationtime)
    config['planet']['planet_pathtospectrum'] = file_destination
    with open('GEMINI/init_simulation.cfg', 'w') as configfile:
        config.write(configfile)
    
    # run LIFEsim through GEMINI
    prompt = "source /dataserver/users/formingworlds/borgmann/.LIFEsim_venv/bin/activate"
    subprocess.run(prompt, shell=True)
    prompt = "python GEMINI/gemini.py"
    subprocess.run(prompt, shell=True)

    # copy GEMINI output to my own output folder
    prompt = "cp GEMINI/outputdir/output-figure.pdf ../output/LIFEsim/observed_spectra/images/" + "observed" + "_" + arcisfolder + "_" + str(int(integrationtime/60/60)) + ".pdf"
    subprocess.run(prompt, shell=True)
    prompt = "cp GEMINI/outputdir/data.txt ../output/LIFEsim/observed_spectra/textfiles/" + "observed" + "_" + arcisfolder + "_" + str(int(integrationtime/60/60))+ ".txt"
    subprocess.run(prompt, shell=True)

In [ ]:
# # convert ARCiS data to be used for LIFEsim
data_path = dataserver_path + '/ARCiS_forwardmodels/'
atmosphere_folders = os.listdir(data_path)

for i in range(len(atmosphere_folders)):
    for integrationtime in integrationtimes:
        arcis_in_lifesim(atmosphere_folders[i],integrationtime) # 4 year mission, 100 planets